In [ ]:
"2025-08-28"
"Testing heater/shaker"
"2028-08-29These code blocks work but the heater does not. Broken locking mechanism. Disassembly required."
"2028-09-03 Disassembled and repaired a hook not attached to a plastic guide on a worm drive."
"Still experiencing problems with lock/unlock"

'2028-09-03 Disassembled and repaired a hook not attached to a plastic guide on a worm drive. '

In [2]:
###############################################################################
# 0) SETUP
###############################################################################
%load_ext autoreload
%autoreload 2
import asyncio

from pylabrobot.liquid_handling import LiquidHandler
from pylabrobot.liquid_handling.backends import STARBackend
from pylabrobot.resources.hamilton import (
    STARLetDeck, MFX_CAR_L5_base, TIP_CAR_480_A00
)
from pylabrobot.resources.hamilton.mfx_modules import Hamilton_MFX_plateholder_DWP_metal_tapped
from pylabrobot.resources.agenbio.plates import AGenBio_1_troughplate_100000uL_Fl
from pylabrobot.resources.corning.plates import Cor_96_wellplate_360ul_Fb
from pylabrobot.resources import HTF, TIP_50ul_w_filter, LTF

# Heater/Shaker
from pylabrobot.heating_shaking import HamiltonHeaterShakerBackend, HeaterShaker
from pylabrobot.resources.coordinate import Coordinate


###############################################################################
# 1) BUILD LH + DECK
###############################################################################
backend = STARBackend()
lh = LiquidHandler(backend=backend, deck=STARLetDeck())
await lh.setup(skip_autoload=True)        # faster while iterating




In [49]:

# ###############################################################################
# # 2) HEATER/SHAKER SETUP (TCC1)
# ###############################################################################
# TCC1 corresponds to index=1 in HamiltonHeaterShakerBackend
hs_backend = HamiltonHeaterShakerBackend(
    index=1,                 # TCC1
    interface=backend        # controlled via the STAR backend
)

heater_shaker = HeaterShaker(
    name="Hamilton HHS SN7590",
    backend=hs_backend,
    size_x=146.2, size_y=103.8, size_z=74.11,
    child_location=Coordinate(x=9.66, y=9.22, z=74.11),
)

# await heater_shaker.setup()

# ###############################################################################
# # 2) HEATER/SHAKER SETUP (auto-detect + robust setup)
# ###############################################################################
import types

# PLR APIs
from pylabrobot.heating_shaking import HamiltonHeaterShakerBackend, HeaterShaker
from pylabrobot.resources.coordinate import Coordinate

# async def detect_hhs_index(star_backend, candidates=(1, 2, 3, 4)):
#     """
#     Try T1..T4 for a responding HHS by sending a harmless 'VP' (version/params) query.
#     Returns the first working index. Raises if none respond.
#     """
#     for idx in candidates:
#         try:
#             resp = await star_backend.send_hhs_command(index=idx, command="VP", read_timeout=1.0)
#             print(f"[HHS detect] T{idx} responded to VP: {resp!r}")
#             return idx
#         except Exception as e:
#             print(f"[HHS detect] T{idx} no response to VP ({e})")
#     raise RuntimeError("No Hamilton Heater Shaker responded on T1..T4. "
#                        "Check cabling to TCC, power, and STAR config.")

# # 2a) Find the T* index (you said TCC1, but we’ll verify so firmware doesn’t throw er55)
# hhs_index = await detect_hhs_index(backend, candidates=(1, 2, 3, 4))
# print(f"[HHS] Using T{hhs_index}")

# # 2b) Build backend/front-end
# hs_backend = HamiltonHeaterShakerBackend(
#     index=hhs_index,       # e.g., 1 if TCC1; our detector confirmed it
#     interface=backend      # control via STAR
# )

# heater_shaker = HeaterShaker(
#     name="Hamilton HHS SN7590",
#     backend=hs_backend,
#     size_x=146.2, size_y=103.8, size_z=74.11,
#     child_location=Coordinate(x=9.66, y=9.22, z=74.11),
# )

# 2c) Try normal setup first. If 'LI' (lock initialize) chokes with er55, bypass it.
try:
    await heater_shaker.setup()
except Exception as e:
    msg = str(e)
    print(f"[HHS] setup() failed: {msg}")

#     # If the only failing part is lock init (common when no adapter/lock present yet),
#     # skip _initialize_lock and retry setup. You can still lock later via hs.lock_plate().
    if "T" in msg and "LI" in msg:
        async def _no_lock_init(self):
            print("[HHS] Skipping lock initialize (LI) during setup.")
            return None
        # monkeypatch this backend instance only
        hs_backend._initialize_lock = types.MethodType(_no_lock_init, hs_backend)

        # Retry setup (now only shaker drive init will run)
        await heater_shaker.setup()
    else:
        # Not a lock-init issue; re-raise so you see the real problem
        raise

# # Optional quick sanity check (raw temp read via firmware and via API)
# try:
#     raw_ts = await backend.send_hhs_command(index=hhs_index, command="TS", read_timeout=1.0)
#     print(f"[HHS] Raw TS response: {raw_ts!r}")
# except Exception as e:
#     print(f"[HHS] TS read failed (raw): {e}")

# print("Middle temp (°C):", await heater_shaker.get_temperature())


[HHS] setup() failed: {'Temperature carrier 1': UnknownHamiltonError('Unknown trace information code 55')}, T1LIid0090er55
[HHS] Skipping lock initialize (LI) during setup.


In [48]:
# ###############################################################################
# # 2d) FUNCTION TESTS (don’t rely on raw TS)
# ###############################################################################

# async def check_temp(label):
#     try:
#         t = await heater_shaker.get_temperature()
#         print(f"[HHS] {label} middle temp: {t:.2f} °C")
#     except Exception as e:
#         print(f"[HHS] get_temperature() failed at {label}: {e}")

# # 1) Baseline read
# await check_temp("baseline")

# # 2) Step down to 30 °C then up to 45 °C, polling to prove control loop works
# for target in (37, 45):
#     print(f"[HHS] Setting temperature to {target} °C …")
#     await heater_shaker.set_temperature(target)
#     # Poll a few times (some firmwares don’t like super-frequent reads)
#     for i in range(6):           # ~30 s total (6 × 5 s); extend if you want
#         await asyncio.sleep(5)
#         await check_temp(f"poll {i+1} @ set {target} °C")
#     print(f"[HHS] Waiting for temperature {target} °C …")
#     # Block until stable at target (backend’s built-in tolerance)
#     await heater_shaker.wait_for_temperature()
#     await check_temp(f"stabilized @ {target} °C")

# 3) Shaking test — safe defaults; works even if lock-init was skipped
print("[HHS] Shaking 600 rpm for 5 s …")
try:
    await heater_shaker.shake(speed=600, direction=0, acceleration=800)
    await asyncio.sleep(5)
    # await heater_shaker.stop_shaking()
    print("[HHS] Shaking test complete.")
except Exception as e:
    print(f"[HHS] shake/stop_shaking failed: {e}")

# 4) Optional: try locking ONLY if an adapter/plate is properly seated
# TRY_LOCK = True   # set True if your smart adapter/plate is in place
# if TRY_LOCK:
#     try:
#         print("[HHS] Trying to lock plate …")
#         await heater_shaker.lock_plate()
#         print("[HHS] Lock engaged. Unlocking …")
#         await heater_shaker.unlock_plate()
#         print("[HHS] Lock/Unlock OK.")
#     except Exception as e:
#         print(f"[HHS] lock/unlock failed (likely adapter/lock not ready): {e}")


[HHS] Shaking 600 rpm for 5 s …
[HHS] shake/stop_shaking failed: {'Temperature carrier 1': UnknownHamiltonError('Unknown trace information code 56')}, T1LPid0089er56


In [ ]:
# # ###############################################################################
# # # 3) SHAKE (lock first, then shake; with a brief retry)
# # ###############################################################################

# import asyncio

# # await heater_shaker.unlock_plate()
# # await heater_shaker.lock_plate()

# async def ensure_lock(hs, attempts=6, delay_s=5):
#     for i in range(attempts):
#         try:
#             await hs.lock_plate()                    # requires smart adapter + plate seated
#             print(f"[HHS] Lock engaged (try {i+1}/{attempts}).")
#             return True
#         except Exception as e:
#             print(f"[HHS] lock_plate() failed (try {i+1}/{attempts}): {e}")
#             await asyncio.sleep(delay_s)
#     return False

# print("[HHS] Preparing to shake: engaging plate lock …")
# if await ensure_lock(heater_shaker):
#     try:
#         # Use a gentle, widely-supported shake profile first
#         rpm = 300       # start low; ramp later once lock is proven stable
#         accel = 800     # rpm/s
#         print(f"[HHS] Shaking {rpm} rpm for 5 s …")
#         await heater_shaker.shake(speed=rpm, direction=0, acceleration=accel)
#         await asyncio.sleep(5)
#         await heater_shaker.stop_shaking()
#         print("[HHS] Shaking test complete.")
#     except Exception as e:
#         print(f"[HHS] shake/stop_shaking failed: {e}")
#     finally:
#         try:
#             await heater_shaker.unlock_plate()
#             print("[HHS] Unlock complete.")
#         except Exception as e:
#             print(f"[HHS] unlock_plate() failed: {e}")
# else:
#     print("[HHS] Could not engage lock after retries. "
#           "Seat the correct smart adapter + plate, then rerun.")


[HHS] Preparing to shake: engaging plate lock …
[HHS] lock_plate() failed (try 1/6): {'Temperature carrier 1': UnknownHamiltonError('Unknown trace information code 56')}, T1LPid0008er56
[HHS] lock_plate() failed (try 2/6): {'Temperature carrier 1': UnknownHamiltonError('Unknown trace information code 56')}, T1LPid0009er56
[HHS] lock_plate() failed (try 3/6): {'Temperature carrier 1': UnknownHamiltonError('Unknown trace information code 56')}, T1LPid0010er56
[HHS] lock_plate() failed (try 4/6): {'Temperature carrier 1': UnknownHamiltonError('Unknown trace information code 56')}, T1LPid0011er56
[HHS] lock_plate() failed (try 5/6): {'Temperature carrier 1': UnknownHamiltonError('Unknown trace information code 56')}, T1LPid0012er56
[HHS] lock_plate() failed (try 6/6): {'Temperature carrier 1': UnknownHamiltonError('Unknown trace information code 56')}, T1LPid0013er56
[HHS] Could not engage lock after retries. Seat the correct smart adapter + plate, then rerun.


In [8]:
###############################################################################
# 2d) FUNCTION TESTS (robust lock + shake)
###############################################################################
# Keep your existing check_temp() from above

async def check_temp(label):
    try:
        t = await heater_shaker.get_temperature()
        print(f"[HHS] {label} middle temp: {t:.2f} °C")
    except Exception as e:
        print(f"[HHS] get_temperature() failed at {label}: {e}")

# --- raw helper for HHS firmware commands over TCC1 ---
async def hhs_raw(cmd: str, *, tag: str = ""):
    try:
        resp = await backend.send_hhs_command(index=1, command=cmd)
        print(f"[HHS RAW]{' '+tag if tag else ''} {cmd!r} -> {resp!r}")
        return resp
    except Exception as e:
        print(f"[HHS RAW]{' '+tag if tag else ''} {cmd!r} FAILED: {e}")
        raise

# --- quick sanity: verify T1 (TCC1) responds ---
# VP = version/probe (just a harmless query)
await hhs_raw("VP", tag="probe T1")    # expect something like 'T1VPid....vpvp'

# --- gentle lock dance: open -> wait -> close, with small backoff ---
async def try_lock(max_tries=6, settle=0.3):
    # ensure we're unlocked first (LP0), small grace, then attempt lock (LP1)
    for i in range(1, max_tries + 1):
        try:
            await hhs_raw("LP0", tag=f"unlock try {i}")
            await asyncio.sleep(settle)
            await hhs_raw("LP1", tag=f"lock try {i}")
            print("[HHS] Plate lock engaged.")
            return True
        except Exception as e:
            print(f"[HHS] lock attempt {i}/{max_tries} failed: {e}")
            await asyncio.sleep(0.6 + 0.2 * i)  # progressive backoff
    return False

# --- optional: read temperature via PLR frontend (works for you) ---
await check_temp("baseline")   # prints middle temp

# --- attempt to lock & shake safely ---
print("[HHS] Preparing to shake: engage plate lock …")
locked = await try_lock()

if not locked:
    print("[HHS] Could not engage lock after retries. " 
          "Check adapter/plate seating (see checklist below).")
else:
    print("[HHS] Lock OK. Setting shake params + spinning …")
    try:
        # conservative starter values; direction 0 = clockwise on HHS
        await heater_shaker.shake(speed=600, direction=0, acceleration=800)
        await asyncio.sleep(5.0)
        await heater_shaker.stop_shaking()
        print("[HHS] Shaking test complete.")
    finally:
        # politely open the clamp when done
        try:
            await hhs_raw("LP0", tag="unlock at end")
        except Exception as e:
            print(f"[HHS] Unlock-at-end failed (non-fatal): {e}")


[HHS RAW] probe T1 'VP' -> 'T1VPid0015vpvp'
[HHS] baseline middle temp: 25.80 °C
[HHS] Preparing to shake: engage plate lock …


Timeout while waiting for response to command T1LP0id0017.


[HHS RAW] unlock try 1 'LP0' FAILED: Timeout while waiting for response to command T1LP0id0017.
[HHS] lock attempt 1/6 failed: Timeout while waiting for response to command T1LP0id0017.


Timeout while waiting for response to command T1LP0id0018.


[HHS RAW] unlock try 2 'LP0' FAILED: Timeout while waiting for response to command T1LP0id0018.
[HHS] lock attempt 2/6 failed: Timeout while waiting for response to command T1LP0id0018.


...

[HHS RAW] unlock try 4 'LP0' FAILED: Timeout while waiting for response to command T1LP0id0020.
[HHS] lock attempt 4/6 failed: Timeout while waiting for response to command T1LP0id0020.


Timeout while waiting for response to command T1LP0id0021.


[HHS RAW] unlock try 5 'LP0' FAILED: Timeout while waiting for response to command T1LP0id0021.
[HHS] lock attempt 5/6 failed: Timeout while waiting for response to command T1LP0id0021.


Timeout while waiting for response to command T1LP0id0022.


[HHS RAW] unlock try 6 'LP0' FAILED: Timeout while waiting for response to command T1LP0id0022.
[HHS] lock attempt 6/6 failed: Timeout while waiting for response to command T1LP0id0022.
[HHS] Could not engage lock after retries. Check adapter/plate seating (see checklist below).


In [ ]:

###############################################################################
# 3) OPTIONAL: ASSIGN TO CARRIER ON DECK
###############################################################################
# # Example: mount the heater/shaker on a P3 shaker carrier at rails=5
# try:
#     from pylabrobot.resources.hamilton import MFX_CAR_P3_SHAKER
#     shaker_carrier = MFX_CAR_P3_SHAKER(
#         name="p3_shaker",
#         modules={1: heater_shaker}   # position slot 1
#     )
#     lh.deck.assign_child_resource(shaker_carrier, rails=5)
# except Exception as e:
#     print("Skipping deck assignment:", e)


# ###############################################################################
# # 4) TEST COMMANDS
# ###############################################################################
# print("Middle temp (°C):", await heater_shaker.get_temperature())
# if hasattr(heater_shaker.backend, "get_edge_temperature"):
#     print("Edge temp (°C):", await heater_shaker.backend.get_edge_temperature())

# await heater_shaker.set_temperature(37)
# await heater_shaker.wait_for_temperature()
# await heater_shaker.lock_plate()
# await heater_shaker.shake(
#     speed=600,
#     direction=0,       # orbital
#     acceleration=1000
# )
# await asyncio.sleep(5)
# await heater_shaker.stop_shaking()
# await heater_shaker.unlock_plate()

# # Clean shutdown
# await heater_shaker.stop()
# await lh.stop()

STARFirmwareError: {'Temperature carrier 1': UnknownHamiltonError('Unknown trace information code 56')}, T1LPid0024er56